# Making a list of subhalos to mock up

The walkthrough mocks up one galaxy, chosen by its subhalo index (`SIDX`). This notebook picks those indices from a snapshot's subhalo catalogue, so you can work through a sample.

We perform the two **example** selections here on the subhalos' stellar mass and on their black holes' total bolometric luminosity. Choose the properties that matter for your project: every subhalo property in the catalogue is read the same way (step 2 lists them), so you can adjust this for your needs in one or two lines in step 4.

1. The snapshot
2. What the catalogue holds
3. The properties
4. Making your selection
5. Plotting the selection criteria
6. Saving the list

Only the catalogue is read, not the particles, so this is quick: about half a minute and 1.5 GB for a full ASTRID snapshot.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from unyt import c, unyt_array

from smock import config
from smock.data import code_mass_to_msun, code_mdot_to_msun_per_yr, open_snapshot

## 1. The snapshot

The test cutout holds only five galaxy groups, so it makes a short list. On Vera the full snapshots are at `config.EXTERNAL["astrid_pig_dir"]` (ASTRID; snapshot 47 is z = 6) and `config.EXTERNAL["ambra_pig_dir"]` (AMBRA); elsewhere, add their paths to `smock/config.py` (see `SETUP.txt`).

In [ ]:
PIG_DIR = config.TEST_CUTOUT_PIG_DIR     # or config.EXTERNAL["astrid_pig_dir"], config.EXTERNAL["ambra_pig_dir"]
SNAP = 47                                # PIG_047_subfind: z = 6 in ASTRID

snap = open_snapshot(PIG_DIR, SNAP)
cat = snap.file                          # catalogue columns are cat["SubGroups/<name>"]
n_sub = cat["SubGroups/SubhaloMassType"].size
print(f"{snap.path.name}: z = {snap.redshift:.2f}, h = {snap.h}, {n_sub} subhalos")

## 2. What the catalogue holds

Each column holds one property for every subhalo, in MP-Gadget's units: masses in 10^10 Msun/h, positions and radii in comoving kpc/h, accretion rates per MP-Gadget time unit (`smock.data` has the conversions). `SubhaloMassType`, `SubhaloLenType` and similar have one column per particle type: 0 gas, 1 dark matter, 4 stars, 5 black holes. Check a column's units against the particles before cutting on it, as was done for the two below.

In [ ]:
print(", ".join(sorted(b.split("/", 1)[1] for b in cat.blocks if b.startswith("SubGroups/"))))

## 3. The properties

Each subhalo's index is its row in the full snapshot's catalogue; that is what the walkthrough's `SIDX` takes. (The test cutout renumbers its subhalos and keeps the full snapshot's indices in `OriginalIndex`.)

In [ ]:
EFFICIENCY = 0.1    # radiative efficiency: L_bol = EFFICIENCY * Mdot c^2 (ASTRID, AMBRA; as in the walkthrough)

if "SubGroups/OriginalIndex" in cat.blocks:
    sidx = cat["SubGroups/OriginalIndex"][:]
else:
    sidx = np.arange(n_sub)

# stellar mass [Msun]: the stars' column (4) of SubhaloMassType
mstar = code_mass_to_msun(cat["SubGroups/SubhaloMassType"][:][:, 4], snap.h)

# bolometric luminosity of all the subhalo's black holes together [erg/s]. SubhaloBHMdot is the
# sum of its BHs' accretion rates (checked against the BH particles), so this is their total,
# not the brightest BH's
mdot = unyt_array(code_mdot_to_msun_per_yr(cat["SubGroups/SubhaloBHMdot"][:]), "Msun/yr")
lbol = (EFFICIENCY * mdot * c**2).to("erg/s").value

print(f"{(mstar > 0).sum()} subhalos with stars, {(lbol > 0).sum()} with an accreting black hole")

## 4. Making your selection

A subhalo is kept if it passes every cut. Set a cut to `None` to switch it off, and add your own below the examples.

In [ ]:
MIN_MSTAR = 1e9      # Msun; None for no cut
MIN_LBOL = None      # erg/s, e.g. 1e43; None for no cut

keep = np.ones(n_sub, dtype=bool)
if MIN_MSTAR is not None:
    keep &= mstar >= MIN_MSTAR
if MIN_LBOL is not None:
    keep &= lbol >= MIN_LBOL

# Your own cuts go here. For example, star-forming galaxies (SubhaloSFR is the sum of the gas
# particles' star formation rates, already in Msun/yr):
# keep &= cat["SubGroups/SubhaloSFR"][:] > 1.0

print(f"{keep.sum()} of {n_sub} subhalos kept")

## 5. Plotting the selection criteria

The subhalo's total black-hole L_bol vs stellar mass, for the subhalos with an accreting black hole. The dashed lines are the cuts.

In [ ]:
show = lbol > 0
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(mstar[show & ~keep], lbol[show & ~keep], s=4, c="0.6", label="not kept", rasterized=True)
ax.scatter(mstar[show & keep], lbol[show & keep], s=8, c="C3", label="kept", rasterized=True)
if MIN_MSTAR is not None:
    ax.axvline(MIN_MSTAR, ls="--", c="k", lw=1)
if MIN_LBOL is not None:
    ax.axhline(MIN_LBOL, ls="--", c="k", lw=1)
ax.set(xscale="log", yscale="log", xlabel=r"$M_\star$ [M$_\odot$]", ylabel=r"$L_{\rm bol}$ of the BHs [erg/s]",
       title=f"{snap.path.name}, z = {snap.redshift:.2f}")
ax.legend()
plt.show()

## 6. Saving the list

The list is a `.npy` file of subhalo indices. To mock up one of them, set the walkthrough's `PIG_DIR` and `SNAP` to this snapshot and `SIDX` to the index, e.g. `SIDX = int(np.load(OUT_FILE)[0])`.

In [ ]:
ORDER = "mass"       # "mass": most massive first; or "shuffled" (with SHUFFLE_SEED), e.g. to spread
                     # the big galaxies evenly when splitting the list over several jobs
SHUFFLE_SEED = 1
LIST_NAME = "mstar_gt_1e9"
OUT_FILE = config.OUTPUT_DIR / "subhalo_lists" / f"PIG_{SNAP:03d}_{LIST_NAME}.npy"

rows = np.flatnonzero(keep)
if ORDER == "mass":
    rows = rows[np.argsort(-mstar[rows], kind="stable")]
else:
    rows = np.random.default_rng(SHUFFLE_SEED).permutation(rows)
subhalo_list = sidx[rows]

OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
np.save(OUT_FILE, subhalo_list)
print(f"{len(subhalo_list)} subhalos -> {OUT_FILE}")
for s, m, l in zip(subhalo_list[:10], mstar[rows][:10], lbol[rows][:10]):
    print(f"  {s:>9d}   log M* = {np.log10(m):.2f}   log L_bol = {np.log10(l) if l > 0 else float('nan'):.2f}")